# Sliding-window ISC analysis cells

This public-release notebook contains no final sample counts, participant IDs, exclusion lists, or study-specific participant membership/overlap tables. Participant dependence is supplied only through local configuration files.

## Acknowledgement and methodological reference

Parts of the ISC code were developed with substantial reference to the [Naturalistic Data Analysis tutorials](https://naturalistic-data.org/content/intro.html). We gratefully acknowledge the authors and contributors of that resource.

**Methodological reference:** Chen, G., Taylor, P. A., Shin, Y. W., Reynolds, R. C., & Cox, R. W. Untangling the relatedness among correlations, Part II: Inter-subject correlation group analysis through linear mixed-effects modeling. NeuroImage 147, 825-840 (2017).


## Cell 1. Set paths and load helper

Keep all participant/dependence files local and do **not** commit them. Two local inputs are used: (1) participant matching IDs for participant-resampling dependencies and (2) a Stage-2 mixed-model design table supplying the identifiers required for crossed-participant random intercepts, repeated-dyad random intercepts, and stimulus/dataset fixed effects.


In [ ]:
from __future__ import annotations

import importlib.util
from pathlib import Path
import sys

CODE_DIR = Path.cwd().resolve()
HELPER_PATH = CODE_DIR / "slidingWindow_isc_functions.py"
if not HELPER_PATH.exists():
    raise FileNotFoundError(f"Cannot find {HELPER_PATH}")

DATA_DIR = CODE_DIR.parents[1] / "data"
RESULTS_DIR = CODE_DIR.parents[1] / "results"
PARTICIPANT_CONFIG = CODE_DIR / "participant_matching_config.json"
MIXED_MODEL_DESIGN = CODE_DIR / "mixed_model_design.csv"
EXAMPLE_ID_CONFIG = CODE_DIR / "participant_matching_config.example.json"
EXAMPLE_MIXED_MODEL_DESIGN = CODE_DIR / "mixed_model_design.example.csv"

spec = importlib.util.spec_from_file_location("slidingWindow_isc_functions", HELPER_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot import {HELPER_PATH}")
sw = importlib.util.module_from_spec(spec)
sys.modules["slidingWindow_isc_functions"] = sw
spec.loader.exec_module(sw)

if not EXAMPLE_ID_CONFIG.exists():
    sw.write_participant_config_template(EXAMPLE_ID_CONFIG)
if not EXAMPLE_MIXED_MODEL_DESIGN.exists():
    EXAMPLE_MIXED_MODEL_DESIGN.write_text(sw.mixed_model_design_template(), encoding="utf-8")

if not PARTICIPANT_CONFIG.exists() or not MIXED_MODEL_DESIGN.exists():
    raise FileNotFoundError(
        "Create the two local configuration files from the schema-only examples, "
        "replace placeholders with local pseudonymous IDs, and keep the local configuration files out of Git."
    )

sw.setup(data_dir=DATA_DIR, results_dir=RESULTS_DIR, participant_config_path=PARTICIPANT_CONFIG)
sw.configure_mixed_model_design(MIXED_MODEL_DESIGN)


### Dependence configuration

The public code does not encode final participant membership or which participants/dyads repeat across datasets. The local participant configuration supplies matching IDs for resampling. The mixed-model CSV supplies, for each Stage-1 dyad coefficient, the two crossed-participant IDs, the repeated-dyad ID, and the stimulus/dataset fixed-effect level. Reuse the same pseudonymous ID whenever the same random-effect unit recurs.


## Cell 2. Check configuration


In [ ]:
sw.show()


## Cell 3. Primary pipeline

Runs the primary temporal-coupling analysis using the dependence configuration above. No participant-level identifiers are written into the public source code.


In [ ]:
primary = sw.run_primary()


## Cell 4. Regional pooled-valence add-on

Run after Cell 3.


In [ ]:
regional_allfive = sw.run_region_allfive()
